# Experiment 4: Multi-Task Validation & Quantization

**Purpose:** Validate segmentation + pose claims (Section 10) and run QAT/PTQ (Section 5).

**Tasks:** Seg, Pose, PTQ, QAT, INT8 export

**Expected time:** ~6-8 hours total

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

import torch, subprocess, os, json
from pathlib import Path
print(f"GPU: {torch.cuda.get_device_name(0)}")

def run_training(cmd, label):
    """Run with real-time output streaming."""
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}\n", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    status = '✅ OK' if proc.returncode == 0 else f'❌ FAILED (code {proc.returncode})'
    print(f"\n  → {label}: {status}\n", flush=True)
    return proc.returncode

## Part A: Segmentation Validation

In [ ]:
# === Cell 2: Train Segmentation ===
run_training(
    "python scripts/train.py --task seg --variant quantized --data coco128.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-q-416-seed42",
    "Segmentation — Quantized"
)
run_training(
    "python scripts/train.py --task seg --variant standard --data coco128.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-std-416-seed42",
    "Segmentation — Standard"
)
print("\n✅ Segmentation training complete!")

## Part B: Pose Estimation Validation

In [ ]:
# === Cell 3: Train Pose ===
run_training(
    "python scripts/train.py --task pose --variant quantized --data coco8-pose.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-q-416-seed42",
    "Pose — Quantized"
)
run_training(
    "python scripts/train.py --task pose --variant standard --data coco8-pose.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-std-416-seed42",
    "Pose — Standard"
)
print("\n✅ Pose estimation training complete!")

In [ ]:
# === Cell 4: Multi-Task Results ===
import glob

print("=" * 70)
print("  Multi-Task Validation Results")
print("=" * 70)

for pattern in ['seg-*-416-seed42', 'pose-*-416-seed42']:
    for f in sorted(glob.glob(f'experiments/results/{pattern}/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        name = Path(f).parent.name
        fm = cfg.get('final_metrics', {})
        print(f"\n  {name}:")
        print(f"    Params:    {cfg.get('params_M', '?')}M")
        for k, v in fm.items():
            if isinstance(v, (int, float)):
                print(f"    {k:15s}: {v:.4f}")

## Part C: Quantization Experiments

In [ ]:
# === Cell 5: PTQ — Post-Training Quantization ===
weights = 'experiments/results/ablation-a4-relu6/best.pt'
if not Path(weights).exists():
    print("Training quick model for quantization...")
    run_training(
        "python scripts/train.py --task det --variant quantized "
        "--imgsz 416 --epochs 50 --seed 42 --warmup 3 --name quant-baseline",
        "Quick baseline for quantization"
    )
    weights = 'experiments/results/quant-baseline/best.pt'

print(f"\nUsing weights: {weights}")

run_training(
    f"python scripts/quantize.py --mode ptq "
    f"--weights {weights} --task det --variant quantized "
    f"--imgsz 416 --n-calib 500 --backend qnnpack",
    "PTQ — Post-Training Quantization"
)

In [ ]:
# === Cell 6: QAT — Quantization-Aware Training ===
run_training(
    f"python scripts/quantize.py --mode qat "
    f"--weights {weights} --task det --variant quantized "
    f"--imgsz 416 --epochs 10 --lr 1e-4 --backend qnnpack",
    "QAT — Quantization-Aware Training"
)

In [ ]:
# === Cell 7: Export Models & Compare Sizes ===
run_training(
    f"python scripts/export.py --weights {weights} "
    f"--task det --variant quantized --imgsz 416 --formats onnx,torchscript",
    "Export FP32 ONNX + TorchScript"
)

print("\n" + "="*60)
print("  Model Sizes")
print("="*60)
for ext in ['*.onnx', '*.torchscript', '*.pt']:
    for f in sorted(Path('experiments').rglob(ext)):
        print(f"  {f.name}: {f.stat().st_size / 1e6:.2f} MB")

In [ ]:
!cd experiments && zip -r /content/multitask_quant_results.zip results/seg-* results/pose-* results/quant-* 2>/dev/null
print("📥 Download: /content/multitask_quant_results.zip")